# **Install necessary libraries**

In [ ]:
# !uv pip install ultralytics
# !uv pip install -U imagecodecs
# import ultralytics
# ultralytics.checks()

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 43.6/235.7 GB disk)


In [ ]:
# Alterntive solution for numpy related dependency problems
!pip uninstall numpy ultralytics torch torchvision -y
!pip cache purge
!pip install numpy==1.24.3
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install ultralytics
!uv pip install -U imagecodecs
!uv pip install -U leafmap
!uv pip install -U localtileserver
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import drive
# from ultralytics import YOLO
# import torch

# **Mount the drive for saving the files**

In [2]:
drive.mount('/content/drive')
root_folder = "/content/drive/MyDrive/Deep-Learning-Building-Extraction-master"

Mounted at /content/drive


In [23]:
import sys
sys.path.append(f"{root_folder}/scripts")


In [25]:
from prediction_to_geodata import yolo_segment_to_shapefile, yolo_obb_to_shapefile

# **Train Instance segmentation**

First check if there is cuda mounted, else change colab runtime to a GPU version. Note that the availability of GPU resource is not always guaranteed for the free subscription. For this training parameters, we will use most default parameters. For experimenting different hyperparamerts, YOLO class train parameters are provided [HERE](https://docs.ultralytics.com/modes/train/#musgd-optimizer) and validation parameters are provided [HERE](https://docs.ultralytics.com/modes/val/#arguments-for-yolo-model-validation)

In [5]:
torch.cuda.is_available()

True

In [ ]:

# define or load a model
# model = YOLO('yolo26.yaml')  # build a new model from scratch using model configuraton file
model = YOLO('yolo26n-seg.pt')  # load a pretrained model (recommended for training)
model.to("cuda" if torch.cuda.is_available() else "cpu")  # send a model to a specific device


results = model.train(data=f"{root_folder}/yolo_dataset/segment/data.yaml",
                      epochs=30,
                      batch=64,
                      project=f'{root_folder}/Results/segment',
                      name='segment',
                      val=False,
                      resume=False)  # freeze=[1,2,3,4,5, 6, 7, 8, 9]
results = model.val( )               # evaluate model performance on the validation set

# **Perform Prediction on test dataset, convert into geospatial file**

In [ ]:
image_folder = f"{root_folder}/raw_dataset/test/images"
output_shapefile = f"{root_folder}/Results/segment/segment_test.geojson"

model = YOLO(f'{root_folder}/Results/segment/segment/weights/best.pt')  # load a pretrained YOLO segmentation model
model.to("cuda" if torch.cuda.is_available() else "cpu")
class_names = {0: 'building'}  # Custom mapping

gdf = yolo_segment_to_shapefile(
    model=model,
    image_folder=image_folder,
    ext='tif',
    output_shapefile=output_shapefile,
    conf_threshold=0.1,
    iou_threshold=0.3,
    class_names=class_names,
    min_area=4,
    simplify_tolerance=1.0
)

# Optional: Display summary statistics
if gdf is not None:
    print("\nSummary Statistics:")
    print(f"Total features: {len(gdf)}")
    print(f"CRS: {gdf.crs}")
    print(f"Columns: {gdf.columns.tolist()}")
    print(f"\nConfidence statistics by class:")
    for class_name in gdf['class_name'].unique():
        class_data = gdf[gdf['class_name'] == class_name]
        print(f"  {class_name}: {len(class_data)} polygons, "
              f"confidence: {class_data['confidence'].mean():.3f} ± {class_data['confidence'].std():.3f}")

# **Visualize segmentation results**

In [11]:
import leafmap
import geopandas as gpd
import kagglehub
import os
data_path = kagglehub.dataset_download("getachewworkineh/kakuma-ceos-training")

In [ ]:

m = leafmap.Map()
m.add_gdf(gdf, layer_name="vector_tile")
raster_file = os.path.join(data_path, "test.tif")
m.add_raster(raster_file, layer_name="raster")
m

# **Train oriented object bounding box detection**

In [ ]:
model = YOLO('yolo26n-obb.pt')  # load a pretrained model (recommended for training)
model.to("cuda" if torch.cuda.is_available() else "cpu")  # send a model to a specific device
# Use the model
results = model.train(data=f"{root_folder}/yolo_dataset/obbox/data.yaml",
                      epochs=30,
                      batch=64,
                      project=f'{root_folder}/Results/obbox',
                      name='obb',
                      val=False,
                      resume=False)
results = model.val()

# **Perform Oriented bounding box inference then to geodata**

In [ ]:

from ultralytics import YOLO
model = YOLO(f'{root_folder}/Results/obbox/obb/weights/best.pt')
model.to("cuda" if torch.cuda.is_available() else "cpu")

image_folder = f"{root_folder}/raw_dataset/test/images"
output_shapefile = f"{root_folder}/Results/obbox/obb_test.geojson"
ext = "tif"
class_names = ["building"]

gdf = yolo_obb_to_shapefile(
        model=model,
        image_folder=image_folder,
        ext=ext,
        output_shapefile=output_shapefile,
        conf_threshold=0.25,
        iou_threshold=0.5,
        class_names=class_names
    )

if gdf is not None:
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    print(gdf[['class_name', 'confidence', 'area', 'angle_deg']].describe())

    # Optional: Save to CSV for easy viewing
    csv_path = output_shapefile.replace('.shp', '_summary.csv')
    gdf.drop('geometry', axis=1).to_csv(csv_path, index=False)
    print(f"\nSummary saved to: {csv_path}")

In [ ]:
m = leafmap.Map()
m.add_gdf(gdf, layer_name="vector_tile")
m.add_raster(raster_file, layer_name="raster")
raster_file = os.path.join(data_path, "test.tif")
m